# Basic Strategy Tutorial: Simple Moving Average Crossover

This notebook demonstrates how to create and backtest a simple moving average crossover strategy using the CBSC Strategy SDK.

## Strategy Overview

The **Moving Average Crossover** strategy is one of the most basic and widely used trading strategies:

- **Buy Signal**: When short-term MA crosses above long-term MA (Golden Cross)
- **Sell Signal**: When short-term MA crosses below long-term MA (Death Cross)

## What You'll Learn

1. Setting up the SDK workspace
2. Fetching historical market data
3. Calculating technical indicators
4. Generating trading signals
5. Running a backtest
6. Analyzing results

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

# Import CBSC Strategy SDK
from cbsc_strategy_sdk import StrategyWorkspace, BacktestAdapter

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Imports complete")

## Step 1: Initialize Workspace

Create a workspace to connect to the CBSC backend and fetch market data.

In [ ]:
# Initialize workspace
workspace = StrategyWorkspace(
    api_base="http://localhost:3003",
    cache_ttl=600,  # Cache for 10 minutes
    timeout=30
)

print("✅ Workspace initialized")

## Step 2: Fetch Historical Data

Download historical price data for Apple (AAPL) stock.

In [ ]:
async def fetch_data():
    """Fetch historical data for AAPL."""
    async with workspace as ws:
        data = await ws.get_historical_data(
            symbol="AAPL",
            start=date(2023, 1, 1),
            end=date(2024, 12, 31),
            interval="1d"
        )
    return data

# Fetch data
data = await fetch_data()

# Display first few rows
print("📊 Historical Data (first 5 rows):")
print(data.head())
print(f"\nTotal rows: {len(data)}")

## Step 3: Calculate Moving Averages

Calculate the short-term (50-day) and long-term (200-day) moving averages.

In [ ]:
# Calculate moving averages
data['SMA_50'] = data['close'].rolling(window=50).mean()
data['SMA_200'] = data['close'].rolling(window=200).mean()

# Display data with moving averages
print("📈 Data with Moving Averages (last 10 rows):")
print(data[['close', 'SMA_50', 'SMA_200']].tail(10))

## Step 4: Generate Trading Signals

Create buy/sell signals based on MA crossovers.

In [ ]:
# Generate signals
data['signal'] = 0

# Buy signal: SMA_50 crosses above SMA_200
data['buy_signal'] = (data['SMA_50'] > data['SMA_200']) & (data['SMA_50'].shift(1) <= data['SMA_200'].shift(1))

# Sell signal: SMA_50 crosses below SMA_200
data['sell_signal'] = (data['SMA_50'] < data['SMA_200']) & (data['SMA_50'].shift(1) >= data['SMA_200'].shift(1))

# Set positions
data.loc[data['SMA_50'] > data['SMA_200'], 'signal'] = 1  # Long
data.loc[data['SMA_50'] < data['SMA_200'], 'signal'] = -1  # Short/Cash

# Count signals
buy_signals = data['buy_signal'].sum()
sell_signals = data['sell_signal'].sum()

print(f"📊 Signal Statistics:")
print(f"  Buy signals: {buy_signals}")
print(f"  Sell signals: {sell_signals}")

## Step 5: Visualize Strategy

Plot price, moving averages, and signals.

In [ ]:
# Plot strategy
plt.figure(figsize=(14, 8))

# Plot price and moving averages
plt.plot(data.index, data['close'], label='Price', alpha=0.7)
plt.plot(data.index, data['SMA_50'], label='SMA 50', alpha=0.7)
plt.plot(data.index, data['SMA_200'], label='SMA 200', alpha=0.7)

# Plot buy signals
buy_dates = data[data['buy_signal']].index
buy_prices = data.loc[data['buy_signal'], 'close']
plt.scatter(buy_dates, buy_prices, color='green', marker='^', s=100, label='Buy', zorder=5)

# Plot sell signals
sell_dates = data[data['sell_signal']].index
sell_prices = data.loc[data['sell_signal'], 'close']
plt.scatter(sell_dates, sell_prices, color='red', marker='v', s=100, label='Sell', zorder=5)

plt.title('Moving Average Crossover Strategy - AAPL')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Chart created")

## Step 6: Run Backtest

Now let's run a formal backtest using the CBSC backtesting engine.

In [ ]:
# Define strategy code
strategy_code = """
def generate_signals(df):
    # Calculate moving averages
    df['SMA_50'] = df['close'].rolling(window=50).mean()
    df['SMA_200'] = df['close'].rolling(window=200).mean()
    
    # Generate signals
    signals = pd.DataFrame(index=df.index)
    signals['signal'] = 0
    signals.loc[df['SMA_50'] > df['SMA_200'], 'signal'] = 1
    signals.loc[df['SMA_50'] < df['SMA_200'], 'signal'] = -1
    
    return signals
"""

# Run backtest
async def run_backtest():
    async with BacktestAdapter() as adapter:
        result = await adapter.run_backtest(
            strategy_code=strategy_code,
            symbols=["AAPL"],
            start_date=date(2023, 1, 1),
            end_date=date(2024, 12, 31),
            initial_capital=100000
        )
    return result

# Execute backtest
result = await run_backtest()

print("✅ Backtest complete")

## Step 7: Analyze Results

Display and analyze backtest performance metrics.

In [ ]:
# Print performance metrics
print("📊 Backtest Results:")
print("=" * 50)
print(f"Total Return: {result.metrics.total_return:.2%}")
print(f"Sharpe Ratio: {result.metrics.sharpe_ratio:.2f}")
print(f"Sortino Ratio: {result.metrics.sortino_ratio:.2f}")
print(f"Max Drawdown: {result.metrics.max_drawdown:.2%}")
print(f"Win Rate: {result.metrics.win_rate:.2%}")
print(f"Profit Factor: {result.metrics.profit_factor:.2f}")
print(f"Total Trades: {result.metrics.total_trades}")
print(f"Avg Trade: ${result.metrics.avg_trade:.2f}")
print(f"Volatility: {result.metrics.volatility:.2%}")

## Step 8: Visualize Performance

Plot equity curve and drawdowns.

In [ ]:
# Plot equity curve
result.plot_equity_curve(figsize=(14, 6), title='MA Crossover Strategy - Equity Curve')

# Plot drawdowns
result.plot_drawdown(figsize=(14, 4), title='MA Crossover Strategy - Drawdown')

print("✅ Performance charts created")

## Step 9: Trade Analysis

Analyze individual trades.

In [ ]:
# Print trade summary
result.print_trade_summary()

# Display recent trades
print("\n📋 Recent Trades:")
for trade in result.trades[-5:]:
    print(f"  {trade.timestamp}: {trade.action} {trade.quantity:.2f} shares @ ${trade.price:.2f}")

## Summary

In this notebook, you learned:

1. ✅ How to initialize the SDK workspace
2. ✅ How to fetch historical market data
3. ✅ How to calculate moving averages
4. ✅ How to generate trading signals
5. ✅ How to run a backtest
6. ✅ How to analyze and visualize results

## Next Steps

- Try different moving average periods (e.g., 20/50, 30/100)
- Add additional indicators (e.g., RSI, MACD)
- Implement stop-loss and take-profit levels
- Test on different symbols and timeframes

## Related Notebooks

- [02_Multi_Factor_Strategy.ipynb](02_Multi_Factor_Strategy.ipynb) - Advanced multi-indicator strategies
- [03_Backtest_Analysis.ipynb](03_Backtest_Analysis.ipynb) - Deep dive into backtest analysis
- [04_Parameter_Optimization.ipynb](04_Parameter_Optimization.ipynb) - Optimize strategy parameters

---
**Version:** 0.1.0  
**Last Updated:** 2026-01-11